# Terrain sources: WorldCover vs NLCD, SRTM vs 3DEP

Sections 1–4 diagnose a specific artifact in **one Broadview tile**. Section 5
asks the prior question across the **whole WSW domain we actually ran**: how
much do these source choices change the terrain the model sees?

The Broadview wind field showed sharp localised dipoles with no terrain
signature. An A/B test tied them to isolated pixels ESA WorldCover labels
**built-up**: the wind field changed **33x more** within 150 m of one than
beyond 1.5 km.

The cause is a classification limit, not the model. WorldCover has a *single*
built-up class covering everything from a farm track to a city centre, and our
lookup maps it to an urban $z_0$ of 0.8 m. Against a grassland background at
0.03 m that is a **3.8σ jump across one 30 m pixel** — a spatial pattern
FuXi-CFD's training set (contiguous Chinese towns) essentially never contains.

NLCD splits development into four intensity classes, so sections 1–4 ask one
question: **what does NLCD call the pixels WorldCover flags as built-up?**


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio

from fuxicfd_wrf.grids import Tile
from fuxicfd_wrf.terrain import nlcd
from fuxicfd_wrf.terrain.build import build_terrain
from fuxicfd_wrf.terrain.roughness import (
    CLASS_NAMES, NLCD_CLASS_NAMES, NLCD_Z0, WORLDCOVER_Z0,
)

TILE  = 'bv_center'
LONLAT = (-103.035, 34.817)
CACHE = Path('/scratch/moptis/fuxicfd/cache')

# NLCD is requester-pays and served as COGs, so we window-read one tile
# (a few hundred KB) rather than the 1.4 GB CONUS mosaic.
import os
os.environ.setdefault('AWS_REQUEST_PAYER', 'requester')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})


## 1. What does NLCD call WorldCover's 'built-up' pixels?


In [ ]:
import pyproj
tf = pyproj.Transformer.from_crs('EPSG:4326', 'EPSG:32613', always_xy=True)
x, y = tf.transform(*LONLAT)
tile = Tile(TILE, x=float(x), y=float(y), crs='EPSG:32613')

wc = np.rint(build_terrain(tile, CACHE, source='worldcover').landcover).astype(int)
nl = np.rint(build_terrain(tile, CACHE, source='nlcd').landcover).astype(int)

bu = wc == 50
print(f"WorldCover calls {bu.sum()} of {wc.size:,} pixels built-up "
      f"({100*bu.mean():.2f}%), all mapped to z0 = {WORLDCOVER_Z0[50]} m\n")
print(f"{'NLCD class':<32}{'px':>6}{'share':>8}{'NLCD z0':>9}")
for c, n in sorted(zip(*np.unique(nl[bu], return_counts=True)), key=lambda p: -p[1]):
    c = int(c)
    print(f'  {NLCD_CLASS_NAMES.get(c, c):<30}{n:6d}{100*n/bu.sum():7.1f}%{NLCD_Z0.get(c, float("nan")):9.3f}')


Three quarters land in **developed, open space** — NLCD's class for verges,
lawns and roadsides, typically <20% impervious, conventional $z_0\approx0.05$ m.
Not 0.8. A further slice is not developed at all.

## 2. Maps: the speckle in context

Two scales, because the problem needs both. The **context** view shows the two
datasets broadly agree on the landscape — this is not wholesale disagreement.
The **zoom** centres on the densest built-up cluster and shows what actually
differs.

Both $z_0$ panels share one scale and one ramp. The scale is **discrete**: $z_0$
comes from a class lookup and takes a handful of values, so a continuous log ramp
would spend its range on empty gaps and render 0.03 and 0.05 — the bulk of the
field — indistinguishable.


In [ ]:
import sys
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))
from plot_roughness_sources import main as plot_sources

out = Path('/scratch/moptis/fuxicfd/out/roughness_sources.png')
plot_sources.__globals__  # noqa: B018 - keep the import obviously used
!python ../scripts/plot_roughness_sources.py --out {out}

from IPython.display import Image, display
display(Image(filename=str(out)))


The vertical dark line in the WorldCover zoom panel is a **road**. NLCD does not
treat it as urban fabric, and the difference panel shows deep blue along exactly
that line — NLCD roughly **16x lower** there (0.8 → 0.05 m).

Broad pale red elsewhere is NLCD reading rather more roughness than WorldCover
across the farmland (shrub/scrub at 0.10 where WorldCover sees grassland at
0.03). That is a real difference too, and unlike the road it is **not**
independently validated here — worth remembering before treating NLCD as truth
everywhere rather than better where it matters.

## 3. Effect on the wind field

From `scripts/ab_terrain.py --sources worldcover nlcd`, same 48 timesteps
through the model with only the roughness channel changed:

| distance to nearest WorldCover built-up px | mean \|Δ wind\| |
|---|---|
| 0–60 m | **0.170 m/s** |
| 60–150 m | 0.111 |
| 150–300 m | 0.062 |
| >1500 m | **0.023** |

**5.2x larger near the old speckle than far from it**, peak 1.04 m/s. The field
std falls 0.109 → 0.107 while the tile mean moves 0.002 m/s: spurious structure
removed, bulk flow untouched.

A caveat worth carrying: the far field moved too (0.023 m/s), so this is a
broader perturbation than a targeted despeckle, which showed 33x. NLCD
reclassifies everywhere, not only at the speckle.

## 4. The open question

FuXi-CFD was **trained on WorldCover-derived roughness**. The model consumes
$z_0$ in metres rather than class codes, so a more accurate $z_0$ should be a
better input — but this does shift the roughness channel relative to training,
and the authors' exact WorldCover→$z_0$ table is unknown, so our mapping already
differs from theirs by an unquantified amount.

`landcover_source` defaults to **`auto`**: NLCD wherever it has coverage
(CONUS/AK/HI), WorldCover elsewhere. Better data at source beats matching
the training set's errors, and the fallback keeps non-US domains running.

That default is not the same as the question being settled. Phase 0 should
still arbitrate it on evidence, the way `dem_mode` was — and section 5 shows
what it has to arbitrate is bigger than the built-up speckle.

`configs/wsw_domain.yaml` pins `worldcover` explicitly. That is **provenance,
not preference**: that store predates the NLCD reader, and leaving it on
`auto` would now silently rebuild it with different terrain under the same
name.


---

## 5. The whole WSW domain

Everything above is one 9 km tile. The question that actually governs the run is
whether the domain we shipped — 238 tiles, 105 × 87 km, 1133 m of relief — has
the same problem, more of it, or none.

`scripts/compare_sources_domain.py` reads all four sources onto one common grid
at the model's own **30 m** (10.15M px) and subtracts them. It compares both
pairs at once:

| channel | used for this run | alternative |
|---|---|---|
| roughness | **ESA WorldCover** 10 m | NLCD 2023, 30 m |
| elevation | **SRTM GL1** 1 arcsec | USGS 3DEP ⅓ arcsec lidar |

The grid is deliberately *not* coarsened for plotting: downsampling first would
average away precisely the isolated single-pixel spikes that motivated the
question.


In [ ]:
# Reading 3DEP over the domain is ~30 s of network for an answer that never
# changes, so the fields are cached to .npz and reused. Drop --reuse to refetch.
NPZ = Path('/scratch/moptis/fuxicfd/out/sources_domain.npz')
FIG = Path('/scratch/moptis/fuxicfd/out/sources_domain.png')

flag = '--reuse' if NPZ.exists() else ''
!python ../scripts/compare_sources_domain.py {flag} --out {NPZ} --fig {FIG}

from IPython.display import Image, display
display(Image(filename=str(FIG)))


### Roughness — the two datasets disagree about the *background*, not just the speckle

```
WorldCover                     NLCD 2023
grassland        90.25%        shrub/scrub            64.87%
tree cover        7.07%        grassland/herbaceous   27.37%
bare/sparse       1.89%        evergreen forest        5.05%
built-up          0.18%        developed, open space   0.94%

mean z0          0.0820 m      mean z0                0.1204 m   (+47%)
```

Two separate findings, and the second is the bigger one.

**1. The built-up speckle is real here, and denser than at Broadview.**
WorldCover flags **18,111 px** as built-up — 0.178% of the domain, but that is
~76 px per tile, comparable to the 83 in the single Broadview tile that produced
the dipoles. What NLCD calls them, though, is *different* from Broadview:

| NLCD class | share | NLCD $z_0$ |
|---|---|---|
| developed, low intensity | 52.3% | 0.25 m |
| developed, open space | 32.7% | 0.05 m |
| shrub/scrub | 6.8% | 0.10 m |
| developed, medium intensity | 6.3% | 0.50 m |

Median $z_0$ NLCD assigns them is **0.25 m against WorldCover's 0.8 m** — a 3.2x
overstatement, not the 16x seen along the Broadview road. More than half are
genuinely low-intensity development. So WSW's built-up pixels are *more real*
than Broadview's, and the correction needed is smaller.

The mechanism still shows up in the isolated-spike count — pixels more than 5x
their own 150 m neighbourhood:

| | isolated $z_0$ spikes |
|---|---|
| WorldCover | 161,978 px (**1.60%**) |
| NLCD | 20,369 px (0.20%) |

**8x more** single-pixel discontinuities under WorldCover. That is the input
pattern that produced the dipoles, and it is present across this domain.

**2. The background disagreement is larger than the speckle and affects everything.**
WorldCover calls 90% of the domain grassland ($z_0$ 0.03 m); NLCD calls 65% of it
shrub/scrub ($z_0$ 0.10 m). Domain-mean $z_0$ is **47% higher** under NLCD. That
is not 0.18% of pixels — it is most of them, and it will move bulk wind speed far
more than the speckle does.

This is the same pale-red background seen in the Broadview context panel, now
measured domain-wide. It is **not independently validated**. NLCD is better where
we checked it (the road), but "better at the speckle" is not evidence for
"correct about the shrub/grass boundary", and a 47% shift in mean roughness is a
big claim to accept untested. This is the thing Phase 0 needs to arbitrate — not
the built-up pixels.

### Elevation — SRTM is fine here, and switching is not worth it

```
                    SRTM GL1        USGS 3DEP
range               1502-2635 m     1504-2631 m
relief              1133 m          1127 m
bias (3DEP-SRTM)    +1.26 m
RMS difference      2.41 m
|diff| p50/p95/max  1.76 / 4.47 / 63.0 m
```

**2.41 m RMS against 1133 m of relief** — 0.2%. The two elevation panels are
visually indistinguishable, which is the honest result. This matches the 2–3.4 m
measured earlier on individual tiles, so that tile-scale number generalises.

The canopy test works as predicted: over NLCD forest (5.06% of the domain) the
bias is **−0.51 m**, over open ground **+1.35 m** — SRTM sits **1.86 m higher
over forest**, exactly the C-band-scatters-inside-the-canopy signature. For wind
that arguably favours SRTM, since the flow sees the canopy top rather than the
soil beneath it.

Two things in the difference panel are worth naming rather than ignoring:

- The **coherent blue triangle in the south-east** is not terrain. It is a 3DEP
  acquisition-block boundary — a different lidar vintage stitched into the
  seamless mosaic. A dataset with visible seams is a poor reason to abandon one
  without them.
- **max \|diff\| 63 m** is a handful of pixels, at cliffs and water bodies where
  a 30 m sample of a 10 m product and a 30 m sample of a 1 arcsec product simply
  land differently.

The one place they genuinely differ is **slope**: mean \|slope\| is 5.03% for
SRTM against 3.83% for 3DEP, correlation 0.831. SRTM is ~31% steeper at the pixel
scale, which is radar speckle noise rather than real terrain. But FuXi-CFD was
**trained on SRTM**, so that noise texture is part of the input statistics it
learned. Feeding it a smoother DEM than it was trained on is a distribution
shift, not obviously an improvement.

### What this changes

- **Elevation: keep SRTM.** 0.2% of relief, trained-on match, and the alternative
  has seams. Closed.
- **Roughness: the open question moved.** It is no longer "does the built-up
  speckle matter" — at WSW that is a 3.2x correction on 0.18% of pixels. It is
  "**is the domain grassland or shrub/scrub**", which is a 47% shift in mean
  $z_0$ across two thirds of the area. Phase 0 on WSW should score that, and the
  ground truth for it is not another land-cover dataset.
